In [1]:
import pandas as pd

In [ ]:
pairs = pd.read_csv("./datas/predict_pairs_12_12_0.4.csv")

In [ ]:
from modules.get_data_table import get_base_data
data = get_base_data()

In [ ]:
from modules.get_data_table import VALUE_PIVOUT, QUANTITY_WEIGHT_PIVOUT
value_pivot = VALUE_PIVOUT(data)
weight_quantity_pivot= QUANTITY_WEIGHT_PIVOUT(data)

value_pivot.head()

In [ ]:
pivot_data = data.groupby(["item_id", "hs4", "ym_seq"], as_index=False)["value"].sum()
pivot_data = pivot_data.pivot(
    index=['item_id', 'hs4'], columns='ym_seq', values='value'
).fillna(0.0)

pivot_weight_data = data.groupby(["item_id", "hs4", "ym_seq"], as_index=False)["quantity_weight"].sum()
pivot_weight_data = pivot_weight_data.pivot(
    index=['item_id', 'hs4'], columns='ym_seq', values='quantity_weight'
).fillna(0.0)

In [ ]:
X_test, y_test = pd.read_csv("./datas/X_test.csv"), pd.read_csv("./datas/y_test.csv")

In [ ]:
from modules.scaler import get_scaler

scaler = get_scaler(X_test.values)
scaled_X_test = scaler.transform(X_test.values)


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=400, max_depth=9, random_state=10)
model.fit(scaled_X_test, y_test.values.ravel())

In [ ]:
import importlib
import modules.create_submit as cs
importlib.reload(cs)

submission = cs.create_submit(value_pivot, weight_quantity_pivot, pairs, data, model, scaler)
print(f"\n✅ 예측 완료! 제출 데이터 크기: {len(submission)}개")
submission.head()

In [ ]:
# import score
# import importlib
# importlib.reload(score)
# from score import print_validation_summary

# y_test = data.loc[data['ym'] == pd.to_datetime('2025-07-01')]
# print_validation_summary(y_test, submission, pairs)

In [ ]:
submission.to_csv('./submit-1.csv', index=False)

In [ ]:
submission.shape